In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# To unzip

In [ ]:
# To unzip
from zipfile import ZipFile
file_name = "/content/processed_data.zip"
with ZipFile(file_name, 'r') as zip:
  zip.extractall()
  print(f'done! {file_name}')

# Lib's

In [ ]:
import cv2
import os
import numpy as np
from datetime import datetime
import time
import glob

# PHASE 1: DATA PREPARATION

In [ ]:
# Data Loading + Color Conversion

def load_and_convert(image_path, color_mode='grayscale', target_size=None):
    """
    Load an image and convert it to the desired color mode.

    Parameters:
    -----------
    image_path : str
        Path to the image file
    color_mode : str
        'grayscale' - Convert to grayscale (default)
        'rgb'       - Convert to RGB
        'bgr'       - Keep as BGR (OpenCV default)
        'unchanged' - Load with alpha channel if present
    target_size : tuple, optional
        (width, height) to resize the image

    Returns:
    --------
    img : numpy.ndarray
        Loaded and converted image
    success : bool
        True if image loaded successfully, False otherwise
    """

    # Check if file exists
    if not os.path.exists(image_path):
        print(f"File not found: {image_path}")
        return None, False

    # Define loading flags based on color_mode
    flag_map = {
        'grayscale'  : cv2.IMREAD_GRAYSCALE,
        'bgr'        : cv2.IMREAD_COLOR,
        'rgb'        : cv2.IMREAD_COLOR,
        'unchanged'  : cv2.IMREAD_UNCHANGED
    }

    flag = flag_map.get(color_mode, cv2.IMREAD_GRAYSCALE)

    # Load image
    img = cv2.imread(image_path, flag)

    # Check if loaded successfully
    if img is None:
        print(f"Failed to load image: {image_path}")
        return None, False

    # Convert RGB if needed
    if color_mode == 'rgb' and len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize if target_size provided
    if target_size:
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)

    return img, True

# SAVE IMAGE

In [ ]:
# SAVE IMAGE
def save_image_step(img, save_dir, step_name, base_name, class_name='Unknown'):
    """
    Save image to step folder with class subfolder.
    """
    # Handle step name with dot (2.5 -> 2_5)
    folder_name = step_name.replace('.', '_')

    # Create class-specific subfolder
    step_folder = os.path.join(save_dir, folder_name, class_name)
    os.makedirs(step_folder, exist_ok=True)

    # Save image
    save_path = os.path.join(step_folder, f"{base_name}.jpg")
    # cv2.imwrite(save_path, img)

    # Also save visualization with overlay
    vis_path = os.path.join(step_folder, f"{base_name}.jpg")
    if len(img.shape) == 2:
        vis_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    else:
        vis_img = img.copy()

    # cv2.putText(vis_img, f"{step_name}", (10, 30),
    #            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.imwrite(vis_path, vis_img)
    time.sleep(0.333)

    return save_path

# Rename and save image

In [ ]:
# Rename and save image
for image_path in glob.glob(r'/content/processed_data/0_original/*/*'):
    # print(f"image_path: {image_path}")
    img, success = load_and_convert(image_path, color_mode='rgb', target_size=(224, 224))
    if success:
        save_dir    = os.path.join('Dataset', 'processed_data')       # r'processed_data'
        step_name   = r'0_original'
        base_name   = datetime.now().strftime("%Y%m%d_%H%M%S_%f")     # r'Meduloblastoma'
        class_name  = os.path.dirname(image_path).split('/')[-1]     # r'Meduloblastoma'
        save_image_step(img, save_dir, step_name, base_name, class_name)
# !rm -rf /content/Dataset

# Data Balancing -> Handle class imbalance

In [ ]:
# Data Balancing -> Handle class imbalance
# Data Augmentation
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import cv2
import glob, os

# Basic augmentation with Keras
def augment_with_keras(image_path, save_dir=r'/content/Dataset/processed_data/1_augmented_images', class_name='Unknown'):
    """Basic image augmentation using Keras"""

    # Create the folder safely
    save_to_dir = os.path.join(save_dir, class_name)
    os.makedirs(save_to_dir, exist_ok=True)

    # Create ImageDataGenerator with various augmentation options
    datagen = ImageDataGenerator(
        rotation_range=10,             # Random rotation (0-40 degrees)
        # width_shift_range=0.15,      # Horizontal shift
        # height_shift_range=0.15,     # Vertical shift
        # shear_range=0.2,             # Shear transformation
        zoom_range=0.2,                # Random zoom
        horizontal_flip=True,          # Horizontal flip
        vertical_flip=True,            # Vertical flip
        brightness_range=[0.5, 1.5],   # Brightness adjustment
        fill_mode='nearest'            # How to fill new pixels
    )

    # Load image
    img = tf.keras.preprocessing.image.load_img(image_path)
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = x.reshape((1,) + x.shape)

    # Generate augmented images
    i = 0
    for batch in datagen.flow(x, batch_size=1,
                              save_to_dir=save_to_dir,
                              save_prefix='aug', save_format='jpeg'):
        i += 1
        if i >= 3:  # Generate 20 augmented images
            break

    # print(f"Generated {i} augmented images in '{save_dir}'")
    return datagen

for image_path in glob.glob(r'/content/Dataset/processed_data/0_original/*/*'):
    class_name = os.path.dirname(image_path).split('/')[-1]
    augment_with_keras(image_path, class_name=class_name)

# Move images to augment_result folder

In [ ]:
# move images to augment_result folder
import shutil
import os

def copy_folder_simple(src, dest):
    """Simple folder copy using shutil."""
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    print(f"Copied: {src} → {dest}")

# Usage
copy_folder_simple(
    r'/content/Dataset/processed_data/0_original/Normal',
    r'/content/Dataset/processed_data/1_augmented_images_result/Normal'
)

copy_folder_simple(
    r'/content/Dataset/processed_data/1_augmented_images/Meduloblastoma',
    r'/content/Dataset/processed_data/1_augmented_images_result/Meduloblastoma'
)


# PHASE 2: NOISE REDUCTION

In [ ]:
# PHASE 2: NOISE REDUCTION
# Gaussian Filter -> Smooth noise (blur)

import cv2
import os

def gaussian_blur_and_save(image_path, save_path, kernel_size=(5, 5), sigma=1.0):
    """
    Apply Gaussian blur to an image and save it.

    Parameters:
    -----------
    image_path : str
        Path to input image
    save_path : str
        Path to save the blurred image (including filename)
    kernel_size : tuple
        Kernel size (width, height) - must be odd numbers
    sigma : float
        Standard deviation of Gaussian kernel

    Returns:
    --------
    blurred_img : numpy.ndarray
        The blurred image
    success : bool
        True if successful, False otherwise
    """

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load: {image_path}")
        return None, False

    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(img, kernel_size, sigma)

    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Save image
    cv2.imwrite(save_path, blurred)
    # print(f"Saved: {save_path}")

    return blurred, True

def batch_gaussian_blur(input_dir, output_dir, kernel_size=(5, 5), sigma=1.0):
    """Apply Gaussian blur to all images in a directory."""

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Process all images
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)

            gaussian_blur_and_save(input_path, output_path, kernel_size, sigma)

    print(f"Batch processing complete!")

# Apply to all Normal images
batch_gaussian_blur(
    input_dir=r'/content/Dataset/processed_data/1_augmented_images_result/Normal',
    output_dir=r'/content/Dataset/processed_data/1_denoised/Normal',
    kernel_size=(5, 5),
    sigma=0.5 # 1.0
)

# Apply to all Medulloblastoma images
batch_gaussian_blur(
    input_dir=r'/content/Dataset/processed_data/1_augmented_images_result/Meduloblastoma',
    output_dir=r'/content/Dataset/processed_data/1_denoised/Meduloblastoma',
    kernel_size=(5, 5),
    sigma=0.5 # 1.0
)

# Batch Image Sharpening

In [ ]:
# Batch Image Sharpening
import cv2
import os

def batch_sharpen_images(input_dir, output_dir, kernel_strength=1.0):
    """
    Apply sharpening to all images in a directory.

    Parameters:
    -----------
    input_dir : str
        Directory containing input images
    output_dir : str
        Directory to save sharpened images
    kernel_strength : float
        Sharpening strength (1.0 = default, >1.0 = stronger)

    Returns:
    --------
    processed_count : int
        Number of images processed
    """

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Define sharpening kernel
    # Base kernel: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]]
    # Strength adjusts the center value
    center = 4 + kernel_strength
    kernel = np.array([
        [0, -1, 0],
        [-1, center, -1],
        [0, -1, 0]
    ])

    processed_count = 0

    # print(f"Sharpening images in: {input_dir}")
    # print("="*60)

    # Process all images
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)

            # Load image
            img = cv2.imread(input_path)
            if img is None:
                print(f"Failed to load: {filename}")
                continue

            # Apply sharpening
            sharpened = cv2.filter2D(img, -1, kernel)

            # Save
            cv2.imwrite(output_path, sharpened)
            processed_count += 1

            # print(f"Sharpened: {filename}")

    # print("="*60)
    # print(f"Complete! Sharpened {processed_count} images")
    # print(f"   Output: {output_dir}")
    print(f"Batch processing complete!")
    # return processed_count

# Sharpen Normal images
batch_sharpen_images(
    input_dir=r'/content/Dataset/processed_data/1_denoised/Normal',
    output_dir=r'/content/Dataset/processed_data/2_sharpened/Normal',
    kernel_strength=1.0
)

# Sharpen Medulloblastoma images
batch_sharpen_images(
    input_dir=r'/content/Dataset/processed_data/1_denoised/Meduloblastoma',
    output_dir=r'/content/Dataset/processed_data/2_sharpened/Meduloblastoma',
    kernel_strength=1.0
)

# Median Filter -> Remove salt-and-pepper noise

In [ ]:
# Median Filter -> Remove salt-and-pepper noise
import cv2
import os
import numpy as np

def batch_median_filter(input_dir, output_dir, kernel_size=3):
    """
    Apply Median filter to all images in a directory.
    Removes salt-and-pepper noise effectively.

    Parameters:
    -----------
    input_dir : str
        Directory containing input images
    output_dir : str
        Directory to save filtered images
    kernel_size : int
        Kernel size (must be odd number, e.g., 3, 5, 7)
        Larger kernel = more noise removal but more blur

    Returns:
    --------
    processed_count : int
        Number of images processed
    """

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Validate kernel size
    if kernel_size % 2 == 0:
        # print(f"Kernel size must be odd. Changing {kernel_size} to {kernel_size + 1}")
        kernel_size += 1

    processed_count = 0

    # print(f"Applying Median Filter to: {input_dir}")
    # print(f"   Kernel size: {kernel_size}x{kernel_size}")
    # print("="*60)

    # Process all images
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)

            # Load image
            img = cv2.imread(input_path)
            if img is None:
                print(f"Failed to load: {filename}")
                continue

            # Apply Median filter
            filtered = cv2.medianBlur(img, kernel_size)

            # Save
            cv2.imwrite(output_path, filtered)
            processed_count += 1

            # print(f"Filtered: {filename}")

    # print("="*60)
    # print(f"Complete! Processed {processed_count} images")
    # print(f"   Output: {output_dir}")
    print(f"Batch processing complete!")
    # return processed_count

# Process Normal images
batch_median_filter(
    input_dir=r'/content/Dataset/processed_data/2_sharpened/Normal',
    output_dir=r'/content/Dataset/processed_data/3_median_filtered/Normal',
    kernel_size=3
)

# Process Medulloblastoma images
batch_median_filter(
    input_dir=r'/content/Dataset/processed_data/2_sharpened/Meduloblastoma',
    output_dir=r'/content/Dataset/processed_data/3_median_filtered/Meduloblastoma',
    kernel_size=3
)

# Bilateral Filter -> Denoise while preserving edges

In [ ]:
# Bilateral Filter -> Denoise while preserving edges

import cv2
import os
import numpy as np

def batch_bilateral_filter(input_dir, output_dir, d=9, sigma_color=75, sigma_space=75):
    """
    Apply Bilateral filter to all images in a directory.
    Denoises while preserving edges (best for medical images).

    Parameters:
    -----------
    input_dir : str
        Directory containing input images
    output_dir : str
        Directory to save filtered images
    d : int
        Diameter of pixel neighborhood (usually 9)
    sigma_color : int
        Filter sigma in color space (larger = more color blending)
    sigma_space : int
        Filter sigma in coordinate space (larger = more spatial blending)

    Returns:
    --------
    processed_count : int
        Number of images processed
    """

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    processed_count = 0

    # print(f"Applying Bilateral Filter to: {input_dir}")
    # print(f"   Parameters: d={d}, sigma_color={sigma_color}, sigma_space={sigma_space}")
    # print("="*60)

    # Process all images
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)

            # Load image
            img = cv2.imread(input_path)
            if img is None:
                print(f"Failed to load: {filename}")
                continue

            # Apply Bilateral filter
            filtered = cv2.bilateralFilter(img, d, sigma_color, sigma_space)

            # Save
            cv2.imwrite(output_path, filtered)
            processed_count += 1

            # print(f"Filtered: {filename}")

    # print("="*60)
    # print(f"Complete! Processed {processed_count} images")
    # print(f"   Output: {output_dir}")
    print(f"Batch processing complete!")
    # return processed_count

# Process Normal images
batch_bilateral_filter(
    input_dir=r'/content/Dataset/processed_data/3_median_filtered/Normal',
    output_dir=r'/content/Dataset/processed_data/4_bilateral_filtered/Normal',
    d=9,
    sigma_color=75,
    sigma_space=75
)

# Process Medulloblastoma images
batch_bilateral_filter(
    input_dir=r'/content/Dataset/processed_data/3_median_filtered/Meduloblastoma',
    output_dir=r'/content/Dataset/processed_data/4_bilateral_filtered/Meduloblastoma',
    d=9,
    sigma_color=75,
    sigma_space=75
)

### Non-Local Means -> Advanced noise reduction

### Wavelet Denoising -> Multi-scale noise removal

### Anisotropic Diffusion -> Edge-preserving smoothing

# Sharpen Normal images

In [ ]:
# Sharpen Normal images
batch_sharpen_images(
    input_dir=r'/content/Dataset/processed_data/4_bilateral_filtered/Normal',
    output_dir=r'/content/Dataset/processed_data/5_sharpened/Normal',
    kernel_strength=1.0
)

# Sharpen Medulloblastoma images
batch_sharpen_images(
    input_dir=r'/content/Dataset/processed_data/4_bilateral_filtered/Meduloblastoma',
    output_dir=r'/content/Dataset/processed_data/5_sharpened/Meduloblastoma',
    kernel_strength=1.0
)

# ANATOMICAL EXTRACTION

In [ ]:
# ANATOMICAL EXTRACTION

import cv2
import numpy as np
import os
from scipy import ndimage
from skimage import measure
import matplotlib.pyplot as plt

# ============================================================
# PHASE 3: ANATOMICAL EXTRACTION
# ============================================================

def batch_anatomical_extraction(input_dir, output_dir, method='combined',
                               threshold=10, padding=10, verbose=True):
    """
    Apply anatomical extraction to all images in a directory.

    Parameters:
    -----------
    input_dir : str
        Directory containing input images
    output_dir : str
        Directory to save extracted images
    method : str
        'skull_strip', 'brain_extract', 'auto_crop', 'combined'
    threshold : int
        Threshold for binary mask (default: 10)
    padding : int
        Padding for cropping (default: 10)
    verbose : bool
        Print progress

    Returns:
    --------
    processed_count : int
        Number of images processed
    """

    os.makedirs(output_dir, exist_ok=True)
    processed_count = 0

    # print(f"Anatomical Extraction: {method}")
    # print(f"   Input: {input_dir}")
    # print("="*60)

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)

            # Load image
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                if verbose:
                    print(f" Failed to load: {filename}")
                continue

            # Apply anatomical extraction
            if method == 'skull_strip':
                result = skull_stripping(img, threshold)
            elif method == 'brain_extract':
                result = brain_extraction(img, threshold)
            elif method == 'auto_crop':
                result = auto_crop(img, padding)
            elif method == 'combined':
                result = full_anatomical_extraction(img, threshold, padding)
            else:
                raise ValueError(f"Unknown method: {method}")

            # Save
            cv2.imwrite(output_path, result)
            processed_count += 1

            # if verbose:
            #     print(f" Processed: {filename}")

    # print("="*60)
    # print(f" Complete! Processed {processed_count} images")
    # print(f"   Output: {output_dir}")

    return processed_count

# ============================================================
# 1. SKULL STRIPPING
# ============================================================

def skull_stripping(img, threshold=10):
    """
    Remove skull/non-brain tissue using thresholding and morphology.

    Parameters:
    -----------
    img : numpy.ndarray
        Input grayscale image
    threshold : int
        Threshold value for binary mask

    Returns:
    --------
    stripped : numpy.ndarray
        Image with skull removed
    mask : numpy.ndarray (optional)
        Brain mask
    """

    # Step 1: Create binary mask
    _, mask = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)

    # Step 2: Morphological operations to clean mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    # Step 3: Fill holes in mask
    mask = fill_holes(mask)

    # Step 4: Apply mask to original image
    stripped = cv2.bitwise_and(img, mask)

    return stripped

# ============================================================
# 2. BRAIN EXTRACTION
# ============================================================

def brain_extraction(img, threshold=10):
    """
    Extract brain tissue (isolate brain parenchyma).

    Parameters:
    -----------
    img : numpy.ndarray
        Input grayscale image
    threshold : int
        Threshold value for binary mask

    Returns:
    --------
    brain : numpy.ndarray
        Extracted brain image
    """

    # Step 1: Create binary mask
    _, mask = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)

    # Step 2: Get largest connected component (brain)
    mask = get_largest_component(mask)

    # Step 3: Fill holes
    mask = fill_holes(mask)

    # Step 4: Apply mask
    brain = cv2.bitwise_and(img, mask)

    return brain

# ============================================================
# 3. AUTO-CROPPING
# ============================================================

def auto_crop(img, padding=10):
    """
    Auto-crop to remove empty borders around the image.

    Parameters:
    -----------
    img : numpy.ndarray
        Input grayscale image
    padding : int
        Padding around the cropped region

    Returns:
    --------
    cropped : numpy.ndarray
        Cropped image
    """

    # Find non-zero pixels
    non_zero_rows = np.any(img > 10, axis=1)
    non_zero_cols = np.any(img > 10, axis=0)

    if not np.any(non_zero_rows) or not np.any(non_zero_cols):
        return img

    # Get bounding box
    y_min = max(0, np.where(non_zero_rows)[0][0] - padding)
    y_max = min(img.shape[0], np.where(non_zero_rows)[0][-1] + padding)
    x_min = max(0, np.where(non_zero_cols)[0][0] - padding)
    x_max = min(img.shape[1], np.where(non_zero_cols)[0][-1] + padding)

    # Crop
    cropped = img[y_min:y_max, x_min:x_max]

    return cropped

# ============================================================
# 4. REGION OF INTEREST (ROI) EXTRACTION
# ============================================================

def extract_roi(img, method='center', size=(200, 200)):
    """
    Extract region of interest (central brain region).

    Parameters:
    -----------
    img : numpy.ndarray
        Input grayscale image
    method : str
        'center' - extract center region
        'largest' - extract largest connected component
    size : tuple
        (width, height) for center extraction

    Returns:
    --------
    roi : numpy.ndarray
        Region of interest
    """

    if method == 'center':
        # Extract center region
        h, w = img.shape
        center_x, center_y = w // 2, h // 2
        half_w, half_h = size[0] // 2, size[1] // 2

        x1 = max(0, center_x - half_w)
        x2 = min(w, center_x + half_w)
        y1 = max(0, center_y - half_h)
        y2 = min(h, center_y + half_h)

        roi = img[y1:y2, x1:x2]

    elif method == 'largest':
        # Extract largest connected component
        _, mask = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
        mask = get_largest_component(mask)
        roi = cv2.bitwise_and(img, mask)

    else:
        raise ValueError(f"Unknown method: {method}")

    return roi

# ============================================================
# 5. CONNECTED COMPONENT ANALYSIS
# ============================================================

def get_largest_component(mask):
    """
    Keep only the largest connected component in the mask.

    Parameters:
    -----------
    mask : numpy.ndarray
        Binary mask

    Returns:
    --------
    largest : numpy.ndarray
        Mask with only the largest component
    """

    # Find connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask, connectivity=8
    )

    if num_labels <= 1:
        return mask

    # Find the largest component (excluding background)
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])

    # Create mask for largest component
    largest = (labels == largest_label).astype(np.uint8) * 255

    return largest

# ============================================================
# 6. MORPHOLOGICAL OPERATIONS
# ============================================================

def morphological_operations(mask, operation='clean', kernel_size=5):
    """
    Apply morphological operations to clean mask.

    Parameters:
    -----------
    mask : numpy.ndarray
        Binary mask
    operation : str
        'clean' - close + open
        'close' - close gaps
        'open' - remove small noise
        'dilate' - expand mask
        'erode' - shrink mask
    kernel_size : int
        Size of kernel

    Returns:
    --------
    cleaned : numpy.ndarray
        Cleaned mask
    """

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

    if operation == 'clean':
        # Close gaps then remove small noise
        cleaned = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel, iterations=1)

    elif operation == 'close':
        cleaned = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    elif operation == 'open':
        cleaned = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)

    elif operation == 'dilate':
        cleaned = cv2.dilate(mask, kernel, iterations=2)

    elif operation == 'erode':
        cleaned = cv2.erode(mask, kernel, iterations=2)

    else:
        raise ValueError(f"Unknown operation: {operation}")

    return cleaned

# ============================================================
# 7. FILL HOLES
# ============================================================

def fill_holes(mask):
    """
    Fill holes in binary mask.

    Parameters:
    -----------
    mask : numpy.ndarray
        Binary mask

    Returns:
    --------
    filled : numpy.ndarray
        Mask with holes filled
    """

    # Find contours
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Create filled mask
    filled = np.zeros_like(mask)
    cv2.drawContours(filled, contours, -1, 255, -1)

    return filled

# ============================================================
# 8. COMPLETE ANATOMICAL EXTRACTION PIPELINE
# ============================================================

def full_anatomical_extraction(img, threshold=10, padding=10):
    """
    Complete anatomical extraction pipeline.

    Steps:
    1. Skull stripping
    2. Brain extraction (largest component)
    3. Morphological cleaning
    4. Auto-cropping

    Parameters:
    -----------
    img : numpy.ndarray
        Input grayscale image
    threshold : int
        Threshold for binary mask
    padding : int
        Padding for cropping

    Returns:
    --------
    result : numpy.ndarray
        Extracted brain image
    """

    # Step 1: Create binary mask
    _, mask = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)

    # Step 2: Get largest component (brain)
    mask = get_largest_component(mask)

    # Step 3: Fill holes
    mask = fill_holes(mask)

    # Step 4: Morphological cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    # Step 5: Apply mask
    extracted = cv2.bitwise_and(img, mask)

    # Step 6: Auto-crop
    extracted = auto_crop(extracted, padding)

    return extracted

# ============================================================
# 9. VISUALIZE ALL METHODS
# ============================================================

def visualize_anatomical_extraction(img_path, output_dir):
    """
    Visualize all anatomical extraction methods on a single image.
    """

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f" Failed to load: {img_path}")
        return

    base_name = os.path.splitext(os.path.basename(img_path))[0]

    # Apply all methods
    methods = {
        'Original': img,
        'Skull Stripped': skull_stripping(img),
        'Brain Extracted': brain_extraction(img),
        'Auto-Cropped': auto_crop(img),
        'ROI Center': extract_roi(img, method='center'),
        'Largest Component': extract_roi(img, method='largest'),
        'Full Pipeline': full_anatomical_extraction(img)
    }

    # Create visualization
    fig, axes = plt.subplots(2, 4, figsize=(16, 10))
    axes = axes.flatten()

    for idx, (name, result) in enumerate(methods.items()):
        if idx < len(axes):
            axes[idx].imshow(result, cmap='gray')
            axes[idx].set_title(f"{name}\n{result.shape}", fontsize=10)
            axes[idx].axis('off')

    # Hide unused subplot
    for idx in range(len(methods), len(axes)):
        axes[idx].axis('off')

    plt.suptitle('Anatomical Extraction Methods Comparison', fontsize=14)
    plt.tight_layout()

    # Save
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{base_name}_comparison.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f" Comparison saved: {save_path}")

    return methods

# ============================================================
# 10. BATCH PROCESSING WITH ALL METHODS
# ============================================================

def batch_all_anatomical_extractions(input_dir, output_base_dir):
    """
    Apply all anatomical extraction methods to all images.
    """

    methods = {
        'skull_stripped': lambda x: skull_stripping(x),
        'brain_extracted': lambda x: brain_extraction(x),
        'auto_cropped': lambda x: auto_crop(x),
        'roi_center': lambda x: extract_roi(x, method='center'),
        'roi_largest': lambda x: extract_roi(x, method='largest'),
        'full_pipeline': lambda x: full_anatomical_extraction(x)
    }

    os.makedirs(output_base_dir, exist_ok=True)

    for method_name, method_func in methods.items():
        output_dir = os.path.join(output_base_dir, method_name)
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n Applying: {method_name}")
        print("="*60)

        processed = 0
        for filename in os.listdir(input_dir):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                input_path = os.path.join(input_dir, filename)
                output_path = os.path.join(output_dir, filename)

                img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue

                result = method_func(img)
                cv2.imwrite(output_path, result)
                processed += 1
                print(f"    {filename}")

        print(f" Complete: {processed} images")

    print("\n" + "="*60)
    print(" All anatomical extraction methods applied!")
    print(f"   Output: {output_base_dir}")

if __name__ == "__main__":
    batch_all_anatomical_extractions(
        input_dir=r'/content/Dataset/processed_data/5_sharpened/Normal',
        output_base_dir=r'/content/Dataset/processed_data/6_anatomical_comparison/Normal'
    )

    batch_all_anatomical_extractions(
        input_dir=r'/content/Dataset/processed_data/5_sharpened/Meduloblastoma',
        output_base_dir=r'/content/Dataset/processed_data/6_anatomical_comparison/Meduloblastoma'
    )

# PHASE 4: SPATIAL NORMALIZATION

In [ ]:
import cv2
import numpy as np
import os
from scipy import ndimage
from skimage import transform
import matplotlib.pyplot as plt

# ============================================================
# PHASE 4: SPATIAL NORMALIZATION
# ============================================================

def batch_spatial_normalization(input_dir, output_dir, method='resize',
                               target_size=(224, 224), verbose=True):
    """
    Apply spatial normalization to all images in a directory.

    Parameters:
    -----------
    input_dir : str
        Directory containing input images
    output_dir : str
        Directory to save normalized images
    method : str
        'resize', 'resample', 'affine', 'register', 'hist_match'
    target_size : tuple
        Target size for resizing (width, height)
    verbose : bool
        Print progress

    Returns:
    --------
    processed_count : int
        Number of images processed
    """

    os.makedirs(output_dir, exist_ok=True)
    processed_count = 0

    print(f" Spatial Normalization: {method}")
    print(f"   Input: {input_dir}")
    print("="*60)

    # Load template for registration (first image)
    template = None
    if method == 'register':
        # Get first image as template
        for filename in os.listdir(input_dir):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                template_path = os.path.join(input_dir, filename)
                template = cv2.imread(template_path, cv2.IMREAD_GRAYSCALE)
                if template is not None:
                    template = cv2.resize(template, target_size)
                    break

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)

            # Load image
            img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                if verbose:
                    print(f" Failed to load: {filename}")
                continue

            # Apply spatial normalization
            if method == 'resize':
                result = resize_image(img, target_size)
            elif method == 'resample':
                result = resample_image(img, target_size)
            elif method == 'affine':
                result = affine_transform(img, target_size)
            elif method == 'register':
                if template is not None:
                    result = register_to_template(img, template)
                else:
                    result = resize_image(img, target_size)
            elif method == 'hist_match':
                result = histogram_matching(img, target_size)
            else:
                raise ValueError(f"Unknown method: {method}")

            # Save
            cv2.imwrite(output_path, result)
            processed_count += 1

            if verbose:
                print(f" Processed: {filename} ({result.shape})")

    print("="*60)
    print(f" Complete! Processed {processed_count} images")
    print(f"   Output: {output_dir}")

    return processed_count


# ============================================================
# 1. RESIZING (Fixed Dimensions)
# ============================================================

def resize_image(img, target_size=(224, 224), interpolation='linear'):
    """
    Resize image to fixed dimensions.

    Parameters:
    -----------
    img : numpy.ndarray
        Input image
    target_size : tuple
        Target size (width, height)
    interpolation : str
        'linear', 'area', 'cubic', 'nearest'

    Returns:
    --------
    resized : numpy.ndarray
        Resized image
    """

    # Choose interpolation method
    interp_map = {
        'linear': cv2.INTER_LINEAR,
        'area': cv2.INTER_AREA,
        'cubic': cv2.INTER_CUBIC,
        'nearest': cv2.INTER_NEAREST
    }

    interp = interp_map.get(interpolation, cv2.INTER_LINEAR)

    # Resize
    resized = cv2.resize(img, target_size, interpolation=interp)

    return resized


# ============================================================
# 2. RESAMPLING (Change pixel spacing)
# ============================================================

def resample_image(img, target_size=(224, 224), method='linear'):
    """
    Resample image to change pixel/voxel spacing.

    Parameters:
    -----------
    img : numpy.ndarray
        Input image
    target_size : tuple
        Target size (width, height)
    method : str
        'linear', 'nearest', 'spline'

    Returns:
    --------
    resampled : numpy.ndarray
        Resampled image
    """

    # Calculate scaling factors
    h, w = img.shape
    scale_y = target_size[1] / h
    scale_x = target_size[0] / w

    # Resample using scipy
    if method == 'linear':
        order = 1
    elif method == 'nearest':
        order = 0
    elif method == 'spline':
        order = 3
    else:
        order = 1

    resampled = ndimage.zoom(img, (scale_y, scale_x), order=order)

    # Convert to uint8
    resampled = resampled.astype(np.uint8)

    return resampled


# ============================================================
# 3. IMAGE REGISTRATION (Align to template)
# ============================================================

def register_to_template(img, template, method='translation'):
    """
    Register image to a template using feature matching or translation.

    Parameters:
    -----------
    img : numpy.ndarray
        Input image
    template : numpy.ndarray
        Template image
    method : str
        'translation' - simple translation
        'feature' - feature-based (SIFT/ORB)
        'affine' - affine transform

    Returns:
    --------
    registered : numpy.ndarray
        Registered image
    """

    # Ensure same size
    if img.shape != template.shape:
        img = cv2.resize(img, (template.shape[1], template.shape[0]))

    if method == 'translation':
        # Simple translation using phase correlation
        # Convert to float for phase correlation
        img_float = img.astype(np.float32)
        template_float = template.astype(np.float32)

        # Compute phase correlation
        shift = cv2.phaseCorrelate(img_float, template_float)
        shift = shift[0]  # (dx, dy)

        # Apply translation
        M = np.float32([[1, 0, shift[0]], [0, 1, shift[1]]])
        registered = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))

    elif method == 'feature':
        # Feature-based registration using ORB
        orb = cv2.ORB_create()

        # Find keypoints and descriptors
        kp1, des1 = orb.detectAndCompute(img, None)
        kp2, des2 = orb.detectAndCompute(template, None)

        if des1 is None or des2 is None:
            return img

        # Match features
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(des1, des2)
        matches = sorted(matches, key=lambda x: x.distance)[:50]

        if len(matches) < 4:
            return img

        # Get matched points
        src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

        # Find homography
        H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

        # Apply transformation
        registered = cv2.warpPerspective(img, H, (img.shape[1], img.shape[0]))

    elif method == 'affine':
        # Affine registration using feature matching
        # Use ORB features
        orb = cv2.ORB_create()
        kp1, des1 = orb.detectAndCompute(img, None)
        kp2, des2 = orb.detectAndCompute(template, None)

        if des1 is None or des2 is None:
            return img

        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(des1, des2)
        matches = sorted(matches, key=lambda x: x.distance)[:50]

        if len(matches) < 4:
            return img

        src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

        # Find affine transform
        M, mask = cv2.estimateAffinePartial2D(src_pts, dst_pts)

        if M is not None:
            registered = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
        else:
            registered = img

    else:
        raise ValueError(f"Unknown method: {method}")

    return registered.astype(np.uint8)


# ============================================================
# 4. AFFINE TRANSFORM (Rotation, Translation, Scaling)
# ============================================================

def affine_transform(img, target_size=None, angle=0, scale=1.0,
                    tx=0, ty=0, method='combined'):
    """
    Apply affine transform (rotation, translation, scaling).

    Parameters:
    -----------
    img : numpy.ndarray
        Input image
    target_size : tuple
        Target size (width, height)
    angle : float
        Rotation angle in degrees
    scale : float
        Scaling factor
    tx : float
        Translation in x direction
    ty : float
        Translation in y direction
    method : str
        'rotation', 'translation', 'scaling', 'combined'

    Returns:
    --------
    transformed : numpy.ndarray
        Transformed image
    """

    h, w = img.shape

    # Set target size
    if target_size is None:
        target_size = (w, h)

    # Center of image
    center = (w // 2, h // 2)

    if method == 'rotation':
        # Rotation only
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        transformed = cv2.warpAffine(img, M, target_size)

    elif method == 'translation':
        # Translation only
        M = np.float32([[1, 0, tx], [0, 1, ty]])
        transformed = cv2.warpAffine(img, M, target_size)

    elif method == 'scaling':
        # Scaling only
        M = cv2.getRotationMatrix2D(center, 0, scale)
        transformed = cv2.warpAffine(img, M, target_size)

    elif method == 'combined':
        # Combined transform
        M = cv2.getRotationMatrix2D(center, angle, scale)
        M[0, 2] += tx
        M[1, 2] += ty
        transformed = cv2.warpAffine(img, M, target_size)

    else:
        raise ValueError(f"Unknown method: {method}")

    return transformed.astype(np.uint8)


# ============================================================
# 5. HISTOGRAM MATCHING
# ============================================================

def histogram_matching(img, target_size=None, template=None):
    """
    Match image intensity distribution to a target/template.

    Parameters:
    -----------
    img : numpy.ndarray
        Input image
    target_size : tuple
        Target size (width, height)
    template : numpy.ndarray
        Template image for histogram matching

    Returns:
    --------
    matched : numpy.ndarray
        Histogram matched image
    """

    # Resize if target_size provided
    if target_size is not None:
        img = cv2.resize(img, target_size)

    # If no template, use uniform distribution
    if template is None:
        # Simple histogram equalization
        matched = cv2.equalizeHist(img)
        return matched

    # Ensure same size
    if img.shape != template.shape:
        template = cv2.resize(template, (img.shape[1], img.shape[0]))

    # Match histogram
    # Flatten images
    img_flat = img.flatten()
    template_flat = template.flatten()

    # Compute cumulative histograms
    img_hist, _ = np.histogram(img_flat, bins=256, range=(0, 256))
    template_hist, _ = np.histogram(template_flat, bins=256, range=(0, 256))

    img_cdf = np.cumsum(img_hist) / np.sum(img_hist)
    template_cdf = np.cumsum(template_hist) / np.sum(template_hist)

    # Create lookup table
    lut = np.zeros(256, dtype=np.uint8)
    for i in range(256):
        # Find closest CDF value
        diff = np.abs(img_cdf[i] - template_cdf)
        lut[i] = np.argmin(diff)

    # Apply LUT
    matched = cv2.LUT(img, lut)

    return matched


# ============================================================
# 6. COMPLETE SPATIAL NORMALIZATION PIPELINE
# ============================================================

def full_spatial_normalization(img, target_size=(224, 224),
                               register=True, template=None):
    """
    Complete spatial normalization pipeline.

    Steps:
    1. Resize to fixed dimensions
    2. Apply affine transform (optional)
    3. Register to template (optional)
    4. Histogram matching (optional)

    Parameters:
    -----------
    img : numpy.ndarray
        Input image
    target_size : tuple
        Target size (width, height)
    register : bool
        Whether to apply registration
    template : numpy.ndarray
        Template for registration

    Returns:
    --------
    normalized : numpy.ndarray
        Normalized image
    """

    # Step 1: Resize
    normalized = resize_image(img, target_size)

    # Step 2: Histogram equalization (optional)
    # normalized = cv2.equalizeHist(normalized)

    # Step 3: Registration (if template provided)
    if register and template is not None:
        template = resize_image(template, target_size)
        normalized = register_to_template(normalized, template, method='translation')

    return normalized


# ============================================================
# 7. VISUALIZE ALL METHODS
# ============================================================

def visualize_spatial_normalization(img_path, output_dir, template_path=None):
    """
    Visualize all spatial normalization methods on a single image.
    """

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f" Failed to load: {img_path}")
        return

    base_name = os.path.splitext(os.path.basename(img_path))[0]

    # Load template if provided
    template = None
    if template_path is not None:
        template = cv2.imread(template_path, cv2.IMREAD_GRAYSCALE)

    # Apply all methods
    methods = {
        'Original': img,
        'Resized (224x224)': resize_image(img, (224, 224)),
        'Resampled': resample_image(img, (224, 224)),
        'Rotated 30°': affine_transform(img, angle=30, method='rotation'),
        'Translated': affine_transform(img, tx=50, ty=30, method='translation'),
        'Scaled 1.5x': affine_transform(img, scale=1.5, method='scaling'),
        'Combined Transform': affine_transform(img, angle=15, scale=1.2, tx=20, ty=10, method='combined'),
        'Histogram Equalized': histogram_matching(img, (224, 224)),
    }

    # Add registration if template exists
    if template is not None:
        registered = register_to_template(img, template)
        methods['Registered'] = registered

    # Create visualization
    n_methods = len(methods)
    n_cols = 4
    n_rows = (n_methods + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten()

    for idx, (name, result) in enumerate(methods.items()):
        if idx < len(axes):
            axes[idx].imshow(result, cmap='gray')
            axes[idx].set_title(f"{name}\n{result.shape}", fontsize=10)
            axes[idx].axis('off')

    # Hide unused subplots
    for idx in range(len(methods), len(axes)):
        axes[idx].axis('off')

    plt.suptitle('Spatial Normalization Methods Comparison', fontsize=14)
    plt.tight_layout()

    # Save
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{base_name}_spatial_comparison.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f" Comparison saved: {save_path}")

    return methods


# ============================================================
# 8. BATCH PROCESSING WITH ALL METHODS
# ============================================================

def batch_all_spatial_normalizations(input_dir, output_base_dir, target_size=(224, 224)):
    """
    Apply all spatial normalization methods to all images.
    """

    methods = {
        'resized': lambda x: resize_image(x, target_size),
        'resampled': lambda x: resample_image(x, target_size),
        'rotated': lambda x: affine_transform(x, target_size, angle=15, method='rotation'),
        'translated': lambda x: affine_transform(x, target_size, tx=20, ty=10, method='translation'),
        'scaled': lambda x: affine_transform(x, target_size, scale=1.2, method='scaling'),
        'hist_eq': lambda x: histogram_matching(x, target_size),
        'registered': lambda x: register_to_template(
            resize_image(x, target_size),
            cv2.resize(template, target_size) if 'template' in locals() else None
        )
    }

    # Load first image as template
    template = None
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            template_path = os.path.join(input_dir, filename)
            template = cv2.imread(template_path, cv2.IMREAD_GRAYSCALE)
            if template is not None:
                template = resize_image(template, target_size)
                break

    os.makedirs(output_base_dir, exist_ok=True)

    for method_name, method_func in methods.items():
        output_dir = os.path.join(output_base_dir, method_name)
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n Applying: {method_name}")
        print("="*60)

        processed = 0
        for filename in os.listdir(input_dir):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                input_path = os.path.join(input_dir, filename)
                output_path = os.path.join(output_dir, filename)

                img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue

                result = method_func(img)
                cv2.imwrite(output_path, result)
                processed += 1
                print(f"    {filename}")

        print(f" Complete: {processed} images")

    print("\n" + "="*60)
    print(" All spatial normalization methods applied!")
    print(f"   Output: {output_base_dir}")

if __name__ == "__main__":
    batch_all_spatial_normalizations(
        input_dir=r'/content/Dataset/processed_data/5_sharpened/Normal',
        output_base_dir=r'/content/Dataset/processed_data/7_spatial_comparison/Normal',
        target_size=(224, 224)
    )

    batch_all_spatial_normalizations(
        input_dir=r'/content/Dataset/processed_data/5_sharpened/Meduloblastoma',
        output_base_dir=r'/content/Dataset/processed_data/7_spatial_comparison/Meduloblastoma',
        target_size=(224, 224)
    )